# Installation and Usage

**1. Clone the Repository**

In [31]:
!git clone https://github.com/Calcifer-0307/Sentiment-Analysis-for-Financial-News.git
%cd Sentiment-Analysis-for-Financial-News

Cloning into 'Sentiment-Analysis-for-Financial-News'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 127 (delta 2), reused 0 (delta 0), pack-reused 118 (from 3)
Receiving objects: 100% (127/127), 62.64 MiB | 27.49 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Sentiment-Analysis-for-Financial-News/Sentiment-Analysis-for-Financial-News/Sentiment-Analysis-for-Financial-News


**2. Set Up Virtual Environment**

In [32]:
!python3 -m venv venv
!source venv/bin/activate

Error: Command '['/content/Sentiment-Analysis-for-Financial-News/Sentiment-Analysis-for-Financial-News/Sentiment-Analysis-for-Financial-News/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: venv/bin/activate: No such file or directory


**3. Install Dependencies**

In [33]:
!pip install -r requirements.txt

**4. Download NLTK Data**

In [35]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

**5.Data Preprocessing**

In [36]:
# Ensure your virtual environment is activated
!python src/preprocess.py

Data shape: (4846, 2)
Label distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
Text length statistics(character level):
count    4846.000000
mean      128.132068
std        56.526180
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: text, dtype: float64
Text length statistics(word level):
count    4846.000000
mean       23.101114
std         9.958474
min         2.000000
25%        16.000000
50%        21.000000
75%        29.000000
max        81.000000
Name: text, dtype: float64
Data shape: (4846, 3)
Label distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
Text length statistics(character level):
count    4846.000000
mean       88.208626
std        43.598506
min         0.000000
25%        55.000000
50%        81.000000
75%       116.000000
max       256.000000
Name: processed_text, dtype: float64
Text length statistics(word l

**6.Loading Data for Training**

In [37]:
from src.data_helper import get_bow_dataloaders, get_tfidf_dataloaders, get_w2c_dataloaders
from src.data_helper import train_bow_or_tfidf_model, train_word2vec_model
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Configuration
train_path = 'data/raw/train_data.csv'
test_path = 'data/raw/test_data.csv'
batch_size = 32

# Example: BoW Model
# 1. Train Vectorizer
bow_model = train_bow_or_tfidf_model([train_path, test_path], CountVectorizer, max_features=5000)

# 2. Get DataLoaders
train_loader, test_loader = get_bow_dataloaders(
    train_path, test_path,
    bow_model,
    batch_size=batch_size
)

# 3. Iterate
for batch_x, batch_y in train_loader:
    # batch_x: (batch_size, 5000)
    # batch_y: (batch_size,)
    pass

# TextCNN for Financial News Sentiment Classification

In [47]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np
import random
from sklearn.metrics import classification_report

**1. Device and random seed**

In [48]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
set_seed(42)

Using device: cpu


**2. Load and preprocess data**

In [49]:
train_df = pd.read_csv('data/raw/train_data.csv')
test_df = pd.read_csv('data/raw/test_data.csv')

# Label mapping
label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
train_df['label'] = train_df['sentiment'].map(label_map)
test_df['label'] = test_df['sentiment'].map(label_map)

# Ensure processed_text column is string (to avoid missing values)
train_df['processed_text'] = train_df['processed_text'].fillna('').astype(str)
test_df['processed_text'] = test_df['processed_text'].fillna('').astype(str)

X_train = train_df['processed_text'].values
y_train = train_df['label'].values
X_test = test_df['processed_text'].values
y_test = test_df['label'].values

print(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")
print("Class distribution in training set:", np.bincount(y_train))

Training set size: 3634, Test set size: 1212
Class distribution in training set: [1022  453 2159]


**3. Build dataset and DataLoader**

In [50]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab=None, max_len=32, is_train=True):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
        self.is_train = is_train
        if is_train:
            self.vocab = self.build_vocab(texts)
        else:
            self.vocab = vocab
        self.vocab_size = len(self.vocab)

    def build_vocab(self, texts, min_freq=2):
        counter = Counter()
        for text in texts:
            text_str = str(text) if not isinstance(text, str) else text
            counter.update(text_str.split())
        vocab = {'<PAD>': 0, '<UNK>': 1}
        for word, freq in counter.items():
            if freq >= min_freq:
                vocab[word] = len(vocab)
        return vocab

    def encode_text(self, text):
        text_str = str(text) if not isinstance(text, str) else text
        tokens = text_str.split()
        ids = [self.vocab.get(token, 1) for token in tokens]
        if len(ids) < self.max_len:
            ids = ids + [0] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return self.encode_text(text), torch.tensor(label, dtype=torch.long)

# Create datasets
train_dataset = TextDataset(X_train, y_train, max_len=32, is_train=True)
test_dataset = TextDataset(X_test, y_test, vocab=train_dataset.vocab, max_len=32, is_train=False)

# DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Vocabulary size: {train_dataset.vocab_size}")


Vocabulary size: 3487


**4. Define TextCNN model**

In [51]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_filters, filter_sizes, num_classes, dropout=0.5):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)               # (batch, seq_len, embed_dim)
        embedded = embedded.permute(0, 2, 1)       # (batch, embed_dim, seq_len)
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
        cat = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(cat)

# Model hyperparameters
vocab_size = train_dataset.vocab_size
embedding_dim = 100
num_filters = 100
filter_sizes = [3, 4, 5]
num_classes = 3
dropout = 0.5

model = TextCNN(vocab_size, embedding_dim, num_filters, filter_sizes, num_classes, dropout).to(device)

# Loss function with class weights to handle imbalance
class_weights = torch.tensor([1.0, 2.5, 1.0]).to(device)   # increase weight for negative class
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

**5. Training and evaluation functions**

In [52]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return total_loss / len(loader), correct / total

**6. Training loop**

In [53]:
epochs = 10
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}")

# ------------------------------
# 7. Final evaluation and classification report
# ------------------------------
def get_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in loader:
            texts = texts.to(device)
            outputs = model(texts)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return all_preds, all_labels

y_pred, y_true = get_predictions(model, test_loader)
target_names = ['positive', 'negative', 'neutral']
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

Epoch 1: Train Loss=1.0366, Test Loss=0.9090, Test Acc=0.6304
Epoch 2: Train Loss=0.8031, Test Loss=0.8230, Test Acc=0.6493
Epoch 3: Train Loss=0.6574, Test Loss=0.7614, Test Acc=0.6815
Epoch 4: Train Loss=0.5297, Test Loss=0.7462, Test Acc=0.6997
Epoch 5: Train Loss=0.4437, Test Loss=0.7277, Test Acc=0.7030
Epoch 6: Train Loss=0.3560, Test Loss=0.7378, Test Acc=0.6964
Epoch 7: Train Loss=0.2964, Test Loss=0.7357, Test Acc=0.6906
Epoch 8: Train Loss=0.2604, Test Loss=0.7496, Test Acc=0.6939
Epoch 9: Train Loss=0.2224, Test Loss=0.8081, Test Acc=0.7038
Epoch 10: Train Loss=0.1959, Test Loss=0.7957, Test Acc=0.6988

Classification Report:
              precision    recall  f1-score   support

    positive       0.59      0.46      0.51       341
    negative       0.57      0.60      0.58       151
     neutral       0.76      0.83      0.80       720

    accuracy                           0.70      1212
   macro avg       0.64      0.63      0.63      1212
weighted avg       0.69      